# 04 - Coop Navigation: MAPPO and parameter sharing

**Setting: cooperative.** *N* agents must cover *N* landmarks. Every step each agent gets
`0.5 x (team reward: minus the sum of distances from each landmark to its closest agent) + 0.5 x (-1 per agent it is bumping into)`.

For a team of identical agents we switch to **MAPPO** (on-policy PPO with a centralised critic). It scales better with the number of
agents and works naturally with shared networks and many parallel environments. Here are the two on the same task:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import *          # load_run, describe_networks, evolution_frames, plot_curves, ...
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

In [ ]:
maddpg3, mappo3 = load_run("spread_maddpg_N3"), load_run("spread_mappo_unshared_N3")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([maddpg3, mappo3], "landmark_dist", ax=ax[0], title="avg distance landmark -> nearest agent")
plot_curves([maddpg3, mappo3], "landmark_dist", ax=ax[1], x="wall_time", title="same, against wall-clock")
plt.tight_layout()

## The MAPPO networks
* **Actor** - outputs a **Beta distribution** for each action number. A Beta lives exactly on [0, 1], the range of MPE actions,
  so no clipping is needed; at the start (alpha = beta) its mean is 0.5 everywhere = zero net force.
* **Critic** - estimates the **value of the situation** V (not of actions), from the global state + which agent it is + time left.
  It is only used during training.

In [ ]:
describe_networks(mappo3)

## Parameter sharing: one network for the whole team
Instead of one actor and one critic per agent, all agents use **the same** actor and critic (they still act on their own observations).
Every agent's experience trains the same network, and all agents are processed in one batch.

In [ ]:
shared3 = load_run("spread_mappo_N3")
describe_networks(shared3)

### Does it learn as well?
Not here. With one shared network the agents look and think alike, so early on they all head for the same places and **huddle in the middle**;
it takes a long time before they split up. Separate networks let each agent specialise quickly.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([shared3, mappo3], "landmark_dist", ax=ax[0], title="avg landmark distance", labels=["shared", "separate"])
plot_curves([shared3, mappo3], "coverage", ax=ax[1], title="landmarks covered", labels=["shared", "separate"])
plt.tight_layout()

In [ ]:
show_video(evolution_frames(shared3, k=4, seed=3), fps=10)      # shared network

In [ ]:
show_video(evolution_frames(mappo3, k=4, seed=3), fps=10)       # separate networks

### What sharing buys: computation
Same data, same algorithm - only the number of networks differs. Measured on a laptop (RTX 3060 GPU, and a single CPU core),
for teams of 3 to 20 agents.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.6))
plot_benchmark(load_benchmark("cuda"), axes=axes[0], title="GPU")
plot_benchmark(load_benchmark("cpu"), axes=axes[1], title="CPU (1 core)")
plt.tight_layout()
for r in load_benchmark("cuda"):
    if r["N"] == "20":
        print(f"N=20, {'shared  ' if r['shared'] == 'True' else 'separate'}: {int(r['params']):>9,} parameters | "
              f"actions {float(r['act_ms']):5.2f} ms | update {float(r['update_s']):.2f} s (GPU)")

* **Parameters** stay small with sharing (they grow only because each observation gets longer with more agents); separate networks grow with N.
* **On a GPU** one big batched pass beats many small ones: choosing actions ~10x and updating ~5x faster at 20 agents, and nearly flat in N.
* **On a single CPU core** the update does the same arithmetic either way, so there is little gain; acting is still faster.


### Scaling the team: 3 -> 4 -> 6 agents
Same settings and training budget, only the team size changes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
for n, colour in ((3, "C0"), (4, "C1"), (6, "C2")):                 # colour = team size
    for name, marker, kind in ((f"spread_mappo_N{n}", "o", "shared"),       # marker = shared / separate
                               (f"spread_mappo_unshared_N{n}", "^", "separate")):
        r = load_run(name)
        ax.plot(r["log"]["step"] / 1e6, r["log"]["landmark_dist"], color=colour, marker=marker, ms=5,
                label=f"N={n}, {kind}")
ax.set_xlabel("environment steps (millions)"); ax.set_ylabel("avg distance landmark -> nearest agent")
ax.set_title("Scaling the team: o shared network, ^ separate networks"); ax.grid(alpha=0.3)
ax.legend(ncol=3, fontsize=8);

With 3 agents the team learns in about half a million steps, and separate networks win clearly. With 4 or 6 agents neither variant
gets far within our budget - but now the **shared network is ahead** (circles below triangles): with more agents, every agent's experience
training the same network starts to pay off. More agents means a team reward
that is harder to attribute to any one agent, many more ways to split the landmarks, and a crowded arena. Ways forward: far more
experience per second (vectorised simulators - next), better credit assignment, curricula that grow the team.

*Side note:* independent PPO (IPPO, each critic sees only its own observation) does **as well as MAPPO** on this task (checked with 3 seeds).
A centralised critic is not automatically better; it mattered most in Speaker-Listener, where the other agent's behaviour was the difficulty.